In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Resolve models directory path safely regardless of CWD
if os.path.basename(os.getcwd()) == 'notebooks':
    models_dir = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
else:
    models_dir = os.path.abspath(os.path.join(os.getcwd(), 'models'))

os.makedirs(models_dir, exist_ok=True)
print(f"Target Models Directory: {models_dir}")



In [ ]:
# Generate Heart Disease Dataset with Specific Clinical Features
np.random.seed(101)
num_records = 2500

ages = np.random.randint(18, 85, size=num_records)
genders = np.random.choice(['male', 'female', 'other'], size=num_records, p=[0.50, 0.46, 0.04])
heights = np.random.normal(170, 10, size=num_records)
weights = np.clip(heights * 0.46 + np.random.normal(0, 11, size=num_records), 40, 150)
bmis = weights / ((heights / 100) ** 2)

smokers = np.random.choice(['yes', 'no'], size=num_records, p=[0.25, 0.75])
alcohol_use = np.random.choice(['low', 'moderate', 'high'], size=num_records, p=[0.55, 0.32, 0.13])
activities = np.random.choice(['sedentary', 'moderate', 'active'], size=num_records, p=[0.40, 0.40, 0.20])
sleep_hours = np.clip(np.random.normal(6.8, 1.3, size=num_records), 4, 10)

systolic = np.clip(115 + (bmis - 22) * 1.5 + (ages - 30) * 0.4 + np.random.normal(0, 10, size=num_records), 85, 210).astype(int)
diastolic = np.clip(72 + (bmis - 22) * 0.9 + (ages - 30) * 0.2 + np.random.normal(0, 7, size=num_records), 55, 125).astype(int)
cholesterol = np.clip(170 + (bmis - 22) * 2.2 + (ages - 30) * 0.9 + np.random.normal(0, 18, size=num_records), 100, 390).astype(int)
glucose = np.clip(85 + (bmis - 22) * 1.2 + np.random.normal(0, 10, size=num_records), 55, 250).astype(int)
insulin = np.clip(5 + (glucose - 85) * 0.12 + np.random.normal(0, 3, size=num_records), 2, 50).astype(int)
heart_rate = np.clip(68 + (bmis - 22) * 0.5 + (ages - 30) * 0.1 + np.random.normal(0, 9, size=num_records), 45, 135).astype(int)

# Target label based on Cardiovascular clinical thresholds
y_heart = ((systolic > 140) | (cholesterol > 240) | ((smokers == 'yes') & (systolic > 130))).astype(int)

df_heart_disease = pd.DataFrame({
    'age': ages, 'gender': genders, 'height': heights, 'weight': weights, 'bmi': bmis,
    'smoking': smokers, 'alcohol': alcohol_use, 'physical_activity': activities,
    'sleep_duration': sleep_hours, 'bp_systolic': systolic, 'bp_diastolic': diastolic,
    'cholesterol': cholesterol, 'glucose': glucose, 'insulin': insulin, 'heart_rate': heart_rate,
    'heart_disease': y_heart
})

csv_path = os.path.join(models_dir, "heart_disease_dataset.csv")
df_heart_disease.to_csv(csv_path, index=False)
print(f"Saved Heart Disease Dataset to: {csv_path} ({len(df_heart_disease)} records, {y_heart.sum()} positive cases)")



In [ ]:
# Preprocessing & Feature Alignment
categorical_cols = ['gender', 'smoking', 'alcohol', 'physical_activity']
numerical_cols = ['age', 'height', 'weight', 'bmi', 'sleep_duration', 'bp_systolic', 'bp_diastolic', 'cholesterol', 'glucose', 'insulin', 'heart_rate']

cat_categories = {
    'gender': ['male', 'female', 'other'],
    'smoking': ['yes', 'no'],
    'alcohol': ['low', 'moderate', 'high'],
    'physical_activity': ['sedentary', 'moderate', 'active']
}

def preprocess_features(df_input):
    X_num = df_input[numerical_cols].values
    encoded_cats = []
    for col, cats in cat_categories.items():
        for category in cats:
            encoded_cats.append((df_input[col] == category).astype(float).values)
    X_cat = np.column_stack(encoded_cats)
    return np.column_stack([X_num, X_cat])

X = preprocess_features(df_heart_disease)
y = df_heart_disease['heart_disease'].values

# Fit and export global StandardScaler
scaler = StandardScaler()
scaler.fit(df_heart_disease[numerical_cols].values)

scaler_path = os.path.join(models_dir, "scaler.pkl")
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
print(f"StandardScaler updated at: {scaler_path}")

# Export Feature Names
feature_names = numerical_cols.copy()
for col, cats in cat_categories.items():
    for category in cats:
        feature_names.append(f"{col}_{category}")

feature_names_path = os.path.join(models_dir, "feature_names.json")
with open(feature_names_path, "w") as f:
    json.dump(feature_names, f, indent=2)
print(f"Feature names saved at: {feature_names_path}")



In [ ]:
# Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numerical features (first 11 columns)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[:, :11] = scaler.transform(X_train[:, :11])
X_test_scaled[:, :11] = scaler.transform(X_test[:, :11])

algorithms = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'decision_tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    'xgboost': XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'svm': SVC(probability=True, C=1.0, kernel='rbf', random_state=42)
}

metrics_file = os.path.join(models_dir, "model_metrics.json")
metrics_report = {}
if os.path.exists(metrics_file):
    try:
        with open(metrics_file, "r") as f:
            metrics_report = json.load(f)
    except Exception:
        metrics_report = {}

metrics_report['heart_disease'] = {}

print(f"\nTraining models for target domain: heart_disease")
for alg_name, clf in algorithms.items():
    clf.fit(X_train_scaled, y_train)
    
    model_filename = os.path.join(models_dir, f"heart_disease_{alg_name}.pkl")
    with open(model_filename, "wb") as f:
        pickle.dump(clf, f)
        
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    metrics_report['heart_disease'][alg_name] = {
        'accuracy': round(float(acc), 4),
        'precision': round(float(prec), 4),
        'recall': round(float(rec), 4),
        'f1_score': round(float(f1), 4),
        'roc_auc': round(float(auc), 4)
    }
    print(f"  [{alg_name}] Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

with open(metrics_file, "w") as f:
    json.dump(metrics_report, f, indent=2)

print(f"\nCompleted training for heart_disease. Updated metrics written to: {metrics_file}")

